In [80]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

# Helper functions
def add_user_message(messages, text):
    user_message = ({"role": "user", "content": text})
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = ({"role": "assistant", "content": text})
    messages.append(assistant_message)

def chat(messages, system=None, effort=None, max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if effort:
        params["output_config"] = {"effort": effort}

    message = client.messages.create(**params)
    # Find the text block instead of assuming content[0] is text
    for block in message.content:
        if block.type == "text":
            return block.text

    return None  # fallback if no text block was found

In [81]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code
* Respond with ONLY the JSON array — no markdown code fences, no explanation, no preamble.

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)

    # Strip fences defensively in case the model adds them anyway
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]

    return json.loads(text.strip())
  

In [82]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [83]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary explanation
* Do not wrap your response in markdown code fences
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [84]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with ONLY the JSON object — no markdown code fences, no explanation, no preamble.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, max_tokens=3000)
    
    # Strip fences defensively in case the model adds them anyway
    eval_text = eval_text.strip()
    if eval_text.startswith("```"):
        eval_text = eval_text.split("```")[1]
        if eval_text.startswith("json"):
            eval_text = eval_text[4:]

    return json.loads(eval_text.strip())

In [85]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [86]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [87]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [88]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.0


In [90]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport boto3\n\ndef generate_presigned_url(bucket_name, object_key):\n    s3_client = boto3.client('s3')\n    url = s3_client.generate_presigned_url(\n        'get_object',\n        Params={'Bucket': bucket_name, 'Key': object_key},\n        ExpiresIn=3600\n    )\n    return url\n```",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and object key, and returns the object's presigned URL valid for 1 hour using boto3.",
      "format": "python",
      "solution_criteria": "Uses boto3 client for 's3', calls generate_presigned_url with ClientMethod 'get_object', includes Bucket and Key params, sets ExpiresIn=3600, returns the URL string."
    },
    "score": 4.5,
    "reasoning": "The solution fully satisfies all specified criteria: it uses the correct boto3 client, method, parameters, and expiration time, and returns the presigned URL as a string. The code is clean and functionally correct for the given task. Minor imp